# Lecture 6 · Exercises

Code for [exercises.md](exercises.md)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

DATA = Path("../../coding_tasks/Data/Out_Data")
rng = np.random.default_rng(1234)

---

## Exercise 6.1 · Causal estimands

In [ ]:
df = pd.read_csv(DATA / "lecture6_exercising.csv")
df["ICE"] = df["Y1"] - df["Y0"]
df

`Y` always equals the potential outcome matching the actual `D`. That is SUTVA holding in the data.

In [ ]:
print("Y matches the assigned potential outcome:", (df["Y"] == np.where(df["D"] == 1, df["Y1"], df["Y0"])).all())

### 2 · Individual causal effects

The `ICE` column above. Four of the twenty are zero or negative.

In [ ]:
print(f"range of ICE: {df['ICE'].min()} to {df['ICE'].max()}")
df["ICE"].value_counts().sort_index().to_frame("count").T

### 3, 4 · ATE and ATT

In [ ]:
treated, control = df[df["D"] == 1], df[df["D"] == 0]

pd.Series(
    {
        "ATE": df["ICE"].mean(),
        "ATT": treated["ICE"].mean(),
        "ATU": control["ICE"].mean(),
    }
).round(3)

In [ ]:
print(f"ATE = {df['ICE'].sum()} / {len(df)} = {df['ICE'].mean()}")
print(f"ATT = {treated['ICE'].sum()} / {len(treated)} = {treated['ICE'].mean()}")

### 5 · Testing the randomization assumptions

If randomization worked, the two groups should be comparable in their potential outcomes, not just the observed ones. Normally impossible to check, but here the full table is given.

In [ ]:
pd.Series(
    {
        "mean Y(0), treated - control": treated["Y0"].mean() - control["Y0"].mean(),
        "mean Y(1), treated - control": treated["Y1"].mean() - control["Y1"].mean(),
    }
).round(3)

The first line is the selection bias from the notes. It is zero, which is why the naive comparison recovers the ATT exactly.

In [ ]:
naive = treated["Y"].mean() - control["Y"].mean()
print(f"naive comparison = {naive}")
print(f"ATT              = {treated['ICE'].mean()}")
print(f"selection bias   = {treated['Y0'].mean() - control['Y0'].mean()}")
print(f"ATE              = {df['ICE'].mean()}")

Note the naive comparison lands on the **ATT**, not the ATE. The decomposition is naive = ATT + selection bias, so zero selection bias gives you the ATT. It estimates the ATE only because randomization makes the two equal in expectation.

---

## Exercise 6.2 · Balance tests

In [ ]:
bal = pd.read_csv(DATA / "lecture6_balance_table.csv")
covariates = [f"X{i}" for i in range(1, 51)]
print(bal.shape)
bal.iloc[:5, :6]

### 1, 2 · t-test for X1

R's `t.test` defaults to Welch, so `equal_var=False` here. R's formula interface computes group 0 minus group 1, giving a negative t. Computing treated minus control flips the sign and changes nothing else.

In [ ]:
t1 = stats.ttest_ind(bal.loc[bal["treatment"] == 1, "X1"],
                     bal.loc[bal["treatment"] == 0, "X1"],
                     equal_var=False)

print(f"mean in group 0 = {bal.loc[bal['treatment'] == 0, 'X1'].mean():.9f}")
print(f"mean in group 1 = {bal.loc[bal['treatment'] == 1, 'X1'].mean():.9f}")
print(f"t statistic     = {t1.statistic:.4f}   (R reports {-t1.statistic:.4f}, group 0 minus group 1)")
print(f"p-value         = {t1.pvalue:.6g}")
print(f"95% CI          = {np.round(t1.confidence_interval(), 4)}")

### 4, 5 · the balance table by hand, and the rejection rates

In [ ]:
def balance_table(data, treat_col, variables):
    rows = []
    for v in variables:
        r = stats.ttest_ind(data.loc[data[treat_col] == 1, v],
                            data.loc[data[treat_col] == 0, v],
                            equal_var=False)
        rows.append(
            {
                "variable": v,
                "mean_treated": data.loc[data[treat_col] == 1, v].mean(),
                "mean_control": data.loc[data[treat_col] == 0, v].mean(),
                "p_value": r.pvalue,
            }
        )
    return pd.DataFrame(rows)


out = balance_table(bal, "treatment", covariates)
out[out["p_value"] < 0.1].round(4)

In [ ]:
for level in [0.10, 0.05]:
    k = (out["p_value"] < level).sum()
    print(f"alpha = {level:.2f}: {k}/{len(out)} rejected = {k / len(out):.0%}  (expected {level:.0%})")

Both rates sit just below what the null predicts, which taken at face value says the randomization worked.

---

## Beyond the exercise · the rejection rate hides a broken randomization

Counting rejections throws away how extreme each p-value is. Two p-values of 1e-14 count the same as two of 0.049.

In [ ]:
low = out.nsmallest(4, "p_value")[["variable", "p_value"]].copy()
low["expected count in 50 tests"] = 50 * low["p_value"]
low

X9 and X28 are what chance produces: running 50 tests, you expect three or four p-values that small. X1 and X2 are not in the same category.

Under a working randomization every p-value is uniform on [0, 1]. Plotting all 50 shows the two outliers that the rejection rate swallows.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(out["p_value"], bins=np.arange(0, 1.05, 0.05), color="#4C78A8", edgecolor="white", linewidth=0.5)
ax.axhline(len(out) * 0.05, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("p-value of the balance test (dashed: what a uniform distribution would give per bin)")
ax.set_ylabel("count")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### What a working randomization actually looks like

Re-assign the treatment at random 500 times, keeping the covariates as they are, and recompute all 50 tests each time. This is the distribution the rejection rates should be judged against.

In [ ]:
X = bal[covariates].to_numpy()
n_treated = int((bal["treatment"] == 1).sum())
REPS = 500

rates_10, rates_05, min_p = [], [], []
for _ in range(REPS):
    d = np.zeros(len(X), dtype=bool)
    d[rng.choice(len(X), size=n_treated, replace=False)] = True
    p = stats.ttest_ind(X[d], X[~d], axis=0, equal_var=False).pvalue
    rates_10.append((p < 0.10).mean())
    rates_05.append((p < 0.05).mean())
    min_p.append(p.min())

rates_10, rates_05, min_p = np.array(rates_10), np.array(rates_05), np.array(min_p)

pd.DataFrame(
    {
        "mean": [rates_10.mean(), rates_05.mean()],
        "observed in the data": [(out["p_value"] < 0.10).mean(), (out["p_value"] < 0.05).mean()],
        "share of re-randomizations at or below the observed": [
            (rates_10 <= (out["p_value"] < 0.10).mean()).mean(),
            (rates_05 <= (out["p_value"] < 0.05).mean()).mean(),
        ],
    },
    index=["rejection rate at 10%", "rejection rate at 5%"],
).round(3)

The observed rejection rates are ordinary. Now the same comparison for the smallest p-value instead of the count.

In [ ]:
print(f"smallest p-value in the data:              {out['p_value'].min():.3g}")
print(f"smallest p-value across {REPS} re-randomizations: {min_p.min():.3g}")
print(f"re-randomizations reaching the observed minimum: {(min_p <= out['p_value'].min()).sum()} of {REPS}")

Not one re-randomization comes anywhere near. The rejection rate says the design is fine, the minimum p-value says the assignment depends on X1 and X2.

Both statistics are computed from the same 50 tests. The difference is that one of them keeps the magnitudes.